# 02 — Data Pipeline: Download, Preprocess & Build Manifests
## TeluguVoiceBridge v2 — Constrained Hardware Plan

This notebook downloads, preprocesses, and builds metadata manifests for all four datasets:

| # | Dataset | Purpose | Raw Size | Processed Size |
|---|---------|---------|----------|----------------|
| 1 | CoVoST-2 Telugu | ASR + Translation | ~3 GB | ~2.5 GB |
| 2 | AI4Bharat Rasa Telugu | Speaker Encoder + ASR | ~2 GB | ~1.5 GB |
| 3 | VCTK | TTS multi-speaker | ~11 GB | ~8 GB |
| 4 | LJSpeech | TTS baseline | ~3 GB | ~2.5 GB |

**Critical rule:** Download ONE dataset → process → delete raw → download next.  
Never have more than one raw dataset on disk simultaneously.

**RAM rule:** Process in batches of 50 files. Call `gc.collect()` between batches.

---
## 2.0 — Common Utilities

In [1]:
import os, gc, shutil, pathlib, csv, json, time, warnings
import numpy as np
import soundfile as sf
import librosa
import torchaudio
import torch
warnings.filterwarnings("ignore")

BASE = pathlib.Path(os.getcwd())  # pipeline_v2/
RAW_DIR = BASE / "data" / "raw"
PROCESSED_DIR = BASE / "data" / "processed"
META_DIR = BASE / "data" / "metadata"
EMB_DIR = BASE / "data" / "embeddings"

# Ensure dirs exist
for d in [RAW_DIR, PROCESSED_DIR / "asr_train", PROCESSED_DIR / "speaker_train",
          PROCESSED_DIR / "tts_train", PROCESSED_DIR / "emotion_train",
          META_DIR, EMB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def disk_free_gb():
    """Return free disk space in GB."""
    return shutil.disk_usage(BASE).free / (1024**3)

def normalize_audio(wav, sr, target_sr=16000):
    """Resample + mono + float32 normalize."""
    if len(wav.shape) > 1:
        wav = wav.mean(axis=0)  # stereo → mono
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    # Loudness normalize to -23 LUFS (simplified peak normalization)
    peak = np.abs(wav).max()
    if peak > 0:
        wav = wav / peak * 0.9  # normalize to -1 dB
    return wav.astype(np.float32)

def save_wav(wav, path, sr=16000):
    """Save audio as 16-bit PCM WAV."""
    sf.write(str(path), wav, sr, subtype="PCM_16")

def check_audio(path):
    """Verify an audio file is readable. Returns duration or None."""
    try:
        info = sf.info(str(path))
        return info.duration
    except Exception:
        return None

print(f"Project root: {BASE}")
print(f"Free disk:    {disk_free_gb():.1f} GB")
print("✓ Utilities loaded.")

Project root: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2
Free disk:    28.2 GB
✓ Utilities loaded.


---
## 2.1 — Dataset 1: CoVoST-2 Telugu (ASR + Translation)

**What it gives us:** Telugu audio + Telugu transcripts + English translations  
**Processing:** Resample to 16 kHz mono, filter 1–20s, save to `asr_train/`  
**Manifest columns:** `audio_path, duration_sec, transcript_telugu, transcript_english, speaker_id, dataset_source, split`

In [2]:
# 2.1.1 — Download CoVoST-2 Telugu via HuggingFace datasets (streaming)
from datasets import load_dataset

print("Loading CoVoST-2 Telugu→English...")
print("(Uses streaming to avoid downloading entire dataset at once)")

# Try canonical ID first, then fallbacks
covost_ds = None
for ds_id in ["facebook/covost2", "covost2", "google/covost2"]:
    try:
        covost_ds = load_dataset(ds_id, "te_en", trust_remote_code=True)
        print(f"  ✓ Loaded from '{ds_id}'")
        break
    except Exception as e:
        print(f"  ✗ '{ds_id}' failed: {e}")
        continue

if covost_ds is None:
    print("\n⚠ CoVoST-2 not available via HuggingFace.")
    print("  Alternative: Use Mozilla Common Voice Telugu + CoVoST-2 TSV files")
    print("  from https://github.com/facebookresearch/covost")
    print("  Will fall back to existing FLEURS data if available.")
else:
    for split in covost_ds:
        print(f"  Split '{split}': {len(covost_ds[split])} samples")
    print("✓ CoVoST-2 loaded.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'facebook/covost2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading CoVoST-2 Telugu→English...
(Uses streaming to avoid downloading entire dataset at once)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'covost2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✗ 'facebook/covost2' failed: Dataset scripts are no longer supported, but found covost2.py


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/covost2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✗ 'covost2' failed: Dataset scripts are no longer supported, but found covost2.py
  ✗ 'google/covost2' failed: Dataset 'google/covost2' doesn't exist on the Hub or cannot be accessed.

⚠ CoVoST-2 not available via HuggingFace.
  Alternative: Use Mozilla Common Voice Telugu + CoVoST-2 TSV files
  from https://github.com/facebookresearch/covost
  Will fall back to existing FLEURS data if available.


In [3]:
# 2.1.2 — Fallback: Use existing FLEURS Telugu data if CoVoST-2 unavailable
# Check for existing FLEURS data from previous pipeline
import glob

FLEURS_DIR = BASE.parent / "data" / "raw" / "fleurs_te_full"
FLEURS_PAIRS = BASE.parent / "data" / "metadata" / "translation_pairs.csv"
FLEURS_MANIFEST = BASE.parent / "data" / "metadata" / "asr_manifest.csv"

use_fleurs_fallback = covost_ds is None

if use_fleurs_fallback:
    # Use rglob to find WAVs in subdirectories (train/, dev/, test/)
    fleurs_wavs = list(FLEURS_DIR.rglob("*.wav")) if FLEURS_DIR.exists() else []
    has_pairs = FLEURS_PAIRS.exists()
    has_manifest = FLEURS_MANIFEST.exists()
    print(f"FLEURS fallback check:")
    print(f"  WAV files:          {len(fleurs_wavs)}")
    print(f"  Translation pairs:  {'Yes' if has_pairs else 'No'}")
    print(f"  ASR manifest:       {'Yes' if has_manifest else 'No'}")
    
    if len(fleurs_wavs) > 0:
        print(f"\n✓ Will use FLEURS Telugu data as CoVoST-2 substitute.")
    else:
        print("\n⚠ No fallback data available. Loading FLEURS from HuggingFace...")
        from datasets import load_dataset
        fleurs_ds = load_dataset("google/fleurs", "te_in", trust_remote_code=True)
        print(f"  ✓ FLEURS Telugu loaded: {sum(len(fleurs_ds[s]) for s in fleurs_ds)} total samples")
        use_fleurs_fallback = True
else:
    print("Using CoVoST-2 (no fallback needed).")

FLEURS fallback check:
  WAV files:          3085
  Translation pairs:  Yes
  ASR manifest:       Yes

✓ Will use FLEURS Telugu data as CoVoST-2 substitute.


In [4]:
# 2.1.3 — Process CoVoST-2 / FLEURS audio → 16kHz mono WAV
import pandas as pd

ASR_OUT = PROCESSED_DIR / "asr_train"
asr_records = []

if not use_fleurs_fallback and covost_ds is not None:
    # ─── Process CoVoST-2 ───
    for split_name in ["train", "validation", "test"]:
        if split_name not in covost_ds:
            continue
        split_data = covost_ds[split_name]
        print(f"\nProcessing CoVoST-2 {split_name}: {len(split_data)} samples")
        
        for i, sample in enumerate(split_data):
            try:
                audio = sample["audio"]
                wav = np.array(audio["array"], dtype=np.float32)
                sr = audio["sampling_rate"]
                
                # Normalize & resample
                wav = normalize_audio(wav, sr, 16000)
                duration = len(wav) / 16000
                
                # Filter by duration
                if duration < 1.0 or duration > 20.0:
                    continue
                
                # Save
                fname = f"covost2_{split_name}_{i:06d}.wav"
                out_path = ASR_OUT / fname
                save_wav(wav, out_path, 16000)
                
                asr_records.append({
                    "audio_path": str(out_path.relative_to(BASE)),
                    "duration_sec": round(duration, 2),
                    "transcript_telugu": sample.get("sentence", ""),
                    "transcript_english": sample.get("translation", ""),
                    "speaker_id": sample.get("client_id", "")[:8],
                    "dataset_source": "covost2",
                    "split": split_name,
                })
            except Exception as e:
                continue
            
            # Memory cleanup every 50 files
            if (i + 1) % 50 == 0:
                gc.collect()
                print(f"  Processed {i+1}/{len(split_data)}...", end="\r")
        
        print(f"  ✓ {split_name}: {sum(1 for r in asr_records if r['split']==split_name)} clips saved")
        gc.collect()

else:
    # ─── Fallback: Process FLEURS Telugu ───
    print("\nProcessing FLEURS Telugu as CoVoST-2 substitute...")
    
    # If we loaded FLEURS from HuggingFace
    if 'fleurs_ds' in dir():
        for split_name in ["train", "validation", "test"]:
            if split_name not in fleurs_ds:
                continue
            split_data = fleurs_ds[split_name]
            print(f"  Processing FLEURS {split_name}: {len(split_data)} samples")
            
            for i, sample in enumerate(split_data):
                try:
                    audio = sample["audio"]
                    wav = np.array(audio["array"], dtype=np.float32)
                    sr = audio["sampling_rate"]
                    wav = normalize_audio(wav, sr, 16000)
                    duration = len(wav) / 16000
                    
                    if duration < 1.0 or duration > 20.0:
                        continue
                    
                    fname = f"fleurs_{split_name}_{i:06d}.wav"
                    out_path = ASR_OUT / fname
                    save_wav(wav, out_path, 16000)
                    
                    asr_records.append({
                        "audio_path": str(out_path.relative_to(BASE)),
                        "duration_sec": round(duration, 2),
                        "transcript_telugu": sample.get("transcription", ""),
                        "transcript_english": "",  # FLEURS doesn't have translations
                        "speaker_id": str(sample.get("id", i)),
                        "dataset_source": "fleurs",
                        "split": split_name if split_name != "validation" else "val",
                    })
                except Exception:
                    continue
                
                if (i + 1) % 50 == 0:
                    gc.collect()
            
            print(f"  ✓ {split_name}: done")
    
    # If existing FLEURS WAVs on disk  
    elif len(fleurs_wavs) > 0 and has_manifest:
        import pandas as pd
        manifest_df = pd.read_csv(FLEURS_MANIFEST)
        print(f"  Using existing manifest: {len(manifest_df)} rows")
        
        for idx, row in manifest_df.iterrows():
            src_path = BASE.parent / row["audio_path"]
            if not src_path.exists():
                continue
            try:
                wav, sr = sf.read(str(src_path))
                wav = normalize_audio(wav, sr, 16000)
                duration = len(wav) / 16000
                if duration < 1.0 or duration > 20.0:
                    continue
                
                fname = f"fleurs_{idx:06d}.wav"
                out_path = ASR_OUT / fname
                save_wav(wav, out_path, 16000)
                
                asr_records.append({
                    "audio_path": str(out_path.relative_to(BASE)),
                    "duration_sec": round(duration, 2),
                    "transcript_telugu": row.get("transcript_telugu", row.get("transcript", "")),
                    "transcript_english": row.get("transcript_english", ""),
                    "speaker_id": str(row.get("speaker_id", "")),
                    "dataset_source": "fleurs",
                    "split": row.get("split", "train"),
                })
            except Exception:
                continue
            
            if (idx + 1) % 50 == 0:
                gc.collect()
        
        print(f"  ✓ Processed {len(asr_records)} FLEURS clips")

print(f"\nTotal ASR records: {len(asr_records)}")
print(f"Free disk: {disk_free_gb():.1f} GB")


Processing FLEURS Telugu as CoVoST-2 substitute...
  Using existing manifest: 3076 rows
  ✓ Processed 2949 FLEURS clips

Total ASR records: 2949
Free disk: 28.3 GB


In [5]:
# 2.1.4 — Build & save ASR manifest
import pandas as pd

asr_df = pd.DataFrame(asr_records)
asr_manifest_path = META_DIR / "asr_manifest.csv"
asr_df.to_csv(asr_manifest_path, index=False)

print(f"ASR manifest saved: {asr_manifest_path}")
print(f"Total rows: {len(asr_df)}")
print(f"\nSplit distribution:")
print(asr_df["split"].value_counts())
print(f"\nDuration stats:")
print(asr_df["duration_sec"].describe())
print(f"\nTotal audio hours: {asr_df['duration_sec'].sum() / 3600:.1f}")
print(f"\nSample rows:")
asr_df.head(3)

ASR manifest saved: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/metadata/asr_manifest.csv
Total rows: 2949

Split distribution:
split
train    2180
test      462
val       307
Name: count, dtype: int64

Duration stats:
count    2949.000000
mean       11.470498
std         3.563408
min         1.440000
25%         8.880000
50%        11.220000
75%        13.920000
max        19.980000
Name: duration_sec, dtype: float64

Total audio hours: 9.4

Sample rows:


,audio_path,duration_sec,transcript_telugu,transcript_english,speaker_id,dataset_source,split
0,data/processed/asr_train/fleurs_000000.wav,7.20,డెంగ్ జియావోపింగ్ నాయకత్వంలో మొదటి ఆర్థిక సంస్...,NaN,nan,fleurs,train
1,data/processed/asr_train/fleurs_000001.wav,13.62,తుఫాను తీరానికి దూరంగా ఉన్నందున యునైటెడ్ స్టేట...,NaN,nan,fleurs,train
2,data/processed/asr_train/fleurs_000002.wav,13.98,ప్రభుతం తరపున దర్యాప్తు చేసేవారు బుధవారం 2 బ్ల...,NaN,nan,fleurs,train


In [6]:
# 2.1.5 — Build translation pairs manifest (for notebook 05)
# Only keep records that have BOTH Telugu and English text

trans_records = [r for r in asr_records 
                 if isinstance(r["transcript_telugu"], str) and r["transcript_telugu"].strip()
                 and isinstance(r["transcript_english"], str) and r["transcript_english"].strip()]

if len(trans_records) > 0:
    trans_df = pd.DataFrame(trans_records)
    trans_manifest_path = META_DIR / "translation_pairs.csv"
    trans_df.to_csv(trans_manifest_path, index=False)
    print(f"Translation pairs manifest saved: {trans_manifest_path}")
    print(f"Total pairs: {len(trans_df)}")
else:
    # Fallback: check for existing translation pairs from previous pipeline
    existing_pairs = BASE.parent / "data" / "metadata" / "translation_pairs.csv"
    if existing_pairs.exists():
        import shutil
        shutil.copy2(existing_pairs, META_DIR / "translation_pairs.csv")
        trans_df = pd.read_csv(META_DIR / "translation_pairs.csv")
        print(f"Copied existing translation pairs: {len(trans_df)} rows")
    else:
        print("⚠ No translation pairs available.")
        print("  Will need CoVoST-2 or manual parallel data for translation fine-tuning.")

Copied existing translation pairs: 1719 rows


In [7]:
# 2.1.6 — Spot-check: verify 10 random processed files
import random

asr_files = list((PROCESSED_DIR / "asr_train").glob("*.wav"))
if len(asr_files) > 0:
    sample_files = random.sample(asr_files, min(10, len(asr_files)))
    ok = 0
    for f in sample_files:
        dur = check_audio(f)
        if dur and 1.0 <= dur <= 20.0:
            ok += 1
        else:
            print(f"  ⚠ Bad file: {f.name} (duration={dur})")
    print(f"Spot check: {ok}/{len(sample_files)} files OK")
else:
    print("No ASR files to check yet.")

# Cleanup CoVoST-2 from memory
if 'covost_ds' in dir():
    del covost_ds
if 'fleurs_ds' in dir():
    del fleurs_ds
gc.collect()
print(f"\nFree disk: {disk_free_gb():.1f} GB")

Spot check: 10/10 files OK

Free disk: 28.3 GB


---
## 2.2 — Dataset 2: AI4Bharat Rasa Telugu (Speaker Encoder + ASR)

**What it gives us:** Studio-quality Telugu speech, multiple speakers  
**Processing:** 16 kHz for ASR, 16 kHz for speaker encoder, 2–8s clips  
**Note:** Rasa may need manual download from ai4bharat.org

In [8]:
# 2.2.1 — Check for Rasa data / Download instructions
RASA_RAW = RAW_DIR / "rasa"
RASA_RAW.mkdir(parents=True, exist_ok=True)

# Check if we already have Rasa data (from previous pipeline or manual download)
rasa_existing = BASE.parent / "data" / "raw" / "rasa"
rasa_wavs = []

for rasa_check in [RASA_RAW, rasa_existing]:
    if rasa_check.exists():
        rasa_wavs = list(rasa_check.rglob("*.wav")) + list(rasa_check.rglob("*.flac"))
        if len(rasa_wavs) > 0:
            print(f"Found {len(rasa_wavs)} Rasa audio files in {rasa_check}")
            RASA_RAW = rasa_check
            break

if len(rasa_wavs) == 0:
    print("Rasa Telugu data not found locally.")
    print("")
    print("Option 1: Download from ai4bharat.org/rasa")
    print(f"  Place files in: {RASA_RAW}")
    print("")
    print("Option 2: Use existing RAVDESS data as speaker encoder substitute")
    
    # Check for RAVDESS fallback
    ravdess_dir = BASE.parent / "data" / "raw" / "ravdess"
    if ravdess_dir.exists():
        rav_wavs = list(ravdess_dir.rglob("*.wav"))
        print(f"  ✓ Found {len(rav_wavs)} RAVDESS files — will use as speaker encoder data")
        RASA_RAW = ravdess_dir
        rasa_wavs = rav_wavs
    else:
        print("  ✗ No RAVDESS fallback either.")
        print("  → Speaker encoder will use pretrained weights only (no fine-tuning).")
else:
    print(f"✓ {len(rasa_wavs)} Rasa/RAVDESS audio files available.")

Rasa Telugu data not found locally.

Option 1: Download from ai4bharat.org/rasa
  Place files in: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/raw/rasa

Option 2: Use existing RAVDESS data as speaker encoder substitute
  ✓ Found 1440 RAVDESS files — will use as speaker encoder data


In [9]:
# 2.2.2 — Process Rasa/RAVDESS for speaker encoder training
SPK_OUT = PROCESSED_DIR / "speaker_train"
spk_records = []
errors_seen = 0

if len(rasa_wavs) > 0:
    print(f"Processing {len(rasa_wavs)} files for speaker encoder...")
    
    for i, wav_path in enumerate(rasa_wavs):
        try:
            wav, sr = sf.read(str(wav_path))
            wav = normalize_audio(wav, sr, 16000)
            duration = len(wav) / 16000
            
            # Speaker encoder needs 2-8 second clips
            if duration < 2.0:
                continue
            
            # Trim to max 8 seconds
            if duration > 8.0:
                wav = wav[:8 * 16000]
                duration = 8.0
            
            # Extract speaker ID from path
            # RAVDESS: Actor_XX/  |  Rasa: speaker_XX/
            parts = wav_path.parts
            speaker_id = "unknown"
            for part in parts:
                if "actor" in part.lower() or "speaker" in part.lower():
                    speaker_id = part
                    break
            if speaker_id == "unknown":
                # Try parent directory name
                speaker_id = wav_path.parent.name
            
            fname = f"spk_{i:06d}.wav"
            out_path = SPK_OUT / fname
            save_wav(wav, out_path, 16000)
            
            spk_records.append({
                "audio_path": str(out_path.relative_to(BASE)),
                "duration_sec": round(duration, 2),
                "speaker_id": speaker_id,
                "dataset_source": "rasa" if "rasa" in str(RASA_RAW).lower() else "ravdess",
            })
        except Exception as e:
            errors_seen += 1
            if errors_seen <= 3:
                print(f"  ERROR on {wav_path.name}: {type(e).__name__}: {e}")
            continue
        
        if (i + 1) % 50 == 0:
            gc.collect()
            print(f"  Processed {i+1}/{len(rasa_wavs)}...", end="\r")
    
    print(f"\n✓ Speaker encoder clips: {len(spk_records)}")
    if errors_seen > 0:
        print(f"  Errors encountered: {errors_seen}")
else:
    print("No speaker training data to process.")

gc.collect()
print(f"Free disk: {disk_free_gb():.1f} GB")

Processing 1440 files for speaker encoder...
  Processed 1400/1440...
✓ Speaker encoder clips: 1435
Free disk: 28.1 GB


In [10]:
# 2.2.3 — Build speaker manifest with train/val/test splits
import pandas as pd

if len(spk_records) > 0:
    spk_df = pd.DataFrame(spk_records)
    
    # Count clips per speaker — need ≥4 for batch construction
    speaker_counts = spk_df["speaker_id"].value_counts()
    valid_speakers = speaker_counts[speaker_counts >= 4].index.tolist()
    spk_df = spk_df[spk_df["speaker_id"].isin(valid_speakers)].copy()
    
    print(f"Speakers with ≥4 clips: {len(valid_speakers)}")
    print(f"Total clips after filtering: {len(spk_df)}")
    
    # Split: 80/10/10 by speaker
    np.random.seed(42)
    speakers = np.array(valid_speakers)
    np.random.shuffle(speakers)
    n = len(speakers)
    train_spks = set(speakers[:int(0.8*n)])
    val_spks = set(speakers[int(0.8*n):int(0.9*n)])
    test_spks = set(speakers[int(0.9*n):])
    
    def assign_split(sid):
        if sid in train_spks: return "train"
        if sid in val_spks: return "val"
        return "test"
    
    spk_df["split"] = spk_df["speaker_id"].apply(assign_split)
    
    spk_manifest_path = META_DIR / "speaker_manifest.csv"
    spk_df.to_csv(spk_manifest_path, index=False)
    print(f"\n✓ Speaker manifest saved: {spk_manifest_path}")
    print(spk_df["split"].value_counts())
else:
    print("No speaker records to save.")

Speakers with ≥4 clips: 24
Total clips after filtering: 1435

✓ Speaker manifest saved: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/metadata/speaker_manifest.csv
split
train    1135
test      180
val       120
Name: count, dtype: int64


---
## 2.3 — Dataset 3: VCTK (TTS Multi-Speaker)

**What it gives us:** 110 English speakers, diverse accents, clean recordings  
**Processing:** Resample 48 kHz → 22050 Hz, mic1 only, loudness normalize  
**Size:** ~11 GB raw → ~8 GB processed

In [11]:
# 2.3.1 — Check for VCTK data / Download
VCTK_RAW = RAW_DIR / "vctk"
VCTK_RAW.mkdir(parents=True, exist_ok=True)

# Check existing locations
vctk_wavs = []
vctk_source = None
for vctk_check in [VCTK_RAW, BASE.parent / "data" / "raw" / "vctk"]:
    if vctk_check.exists():
        vctk_wavs = list(vctk_check.rglob("*.wav")) + list(vctk_check.rglob("*.flac"))
        if len(vctk_wavs) > 0:
            vctk_source = vctk_check
            break

if len(vctk_wavs) == 0:
    print("VCTK not found locally. Attempting HuggingFace download...")
    try:
        from datasets import load_dataset
        vctk_ds = load_dataset("vctk", split="train", trust_remote_code=True)
        print(f"  ✓ VCTK loaded from HuggingFace: {len(vctk_ds)} samples")
    except Exception as e:
        print(f"  ✗ HuggingFace VCTK failed: {e}")
        print("")
        print("Manual download instructions:")
        print("  1. Go to: https://datashare.ed.ac.uk/handle/10283/3443")
        print("  2. Download VCTK-Corpus-0.92.zip")
        print(f"  3. Extract to: {VCTK_RAW}")
        print("  4. Re-run this cell")
        vctk_ds = None
else:
    vctk_ds = None
    # Filter to mic1 only (VCTK has duplicate mic1/mic2)
    mic1_wavs = [w for w in vctk_wavs if "mic2" not in str(w).lower()]
    if len(mic1_wavs) < len(vctk_wavs):
        print(f"Filtered mic2: {len(vctk_wavs)} → {len(mic1_wavs)} (mic1 only)")
        vctk_wavs = mic1_wavs
    print(f"✓ VCTK: {len(vctk_wavs)} files from {vctk_source}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'vctk' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


VCTK not found locally. Attempting HuggingFace download...


README.md: 0.00B [00:00, ?B/s]

vctk.py: 0.00B [00:00, ?B/s]

  ✗ HuggingFace VCTK failed: Dataset scripts are no longer supported, but found vctk.py

Manual download instructions:
  1. Go to: https://datashare.ed.ac.uk/handle/10283/3443
  2. Download VCTK-Corpus-0.92.zip
  3. Extract to: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/raw/vctk
  4. Re-run this cell


In [12]:
# 2.3.2 — Process VCTK → 22050 Hz mono WAV
TTS_OUT = PROCESSED_DIR / "tts_train"
tts_records = []

# Load VCTK text transcripts if available
vctk_texts = {}  # {filename_stem: transcript}
if vctk_source:
    for txt_file in vctk_source.rglob("*.txt"):
        try:
            text = txt_file.read_text().strip()
            vctk_texts[txt_file.stem] = text
        except Exception:
            pass

if vctk_ds is not None:
    # ─── Process from HuggingFace dataset ───
    print(f"Processing {len(vctk_ds)} VCTK samples...")
    for i, sample in enumerate(vctk_ds):
        try:
            audio = sample["audio"]
            wav = np.array(audio["array"], dtype=np.float32)
            sr = audio["sampling_rate"]
            wav = normalize_audio(wav, sr, 22050)
            duration = len(wav) / 22050
            
            if duration < 1.0 or duration > 10.0:
                continue
            
            speaker_id = sample.get("speaker_id", f"spk_{i}")
            transcript = sample.get("text", "")
            
            fname = f"vctk_{speaker_id}_{i:06d}.wav"
            out_path = TTS_OUT / fname
            save_wav(wav, out_path, 22050)
            
            tts_records.append({
                "audio_path": str(out_path.relative_to(BASE)),
                "duration_sec": round(duration, 2),
                "transcript": transcript,
                "speaker_id": speaker_id,
                "dataset_source": "vctk",
            })
        except Exception:
            continue
        
        if (i + 1) % 50 == 0:
            gc.collect()
            print(f"  Processed {i+1}/{len(vctk_ds)}...", end="\r")
    
    del vctk_ds
    gc.collect()

elif len(vctk_wavs) > 0:
    # ─── Process from local files ───
    print(f"Processing {len(vctk_wavs)} VCTK files...")
    for i, wav_path in enumerate(vctk_wavs):
        try:
            wav, sr = sf.read(str(wav_path))
            wav = normalize_audio(wav, sr, 22050)
            duration = len(wav) / 22050
            
            if duration < 1.0 or duration > 10.0:
                continue
            
            # Extract speaker ID from path (e.g., p225/p225_001.wav)
            speaker_id = wav_path.parent.name
            transcript = vctk_texts.get(wav_path.stem, "")
            
            fname = f"vctk_{speaker_id}_{i:06d}.wav"
            out_path = TTS_OUT / fname
            save_wav(wav, out_path, 22050)
            
            tts_records.append({
                "audio_path": str(out_path.relative_to(BASE)),
                "duration_sec": round(duration, 2),
                "transcript": transcript,
                "speaker_id": speaker_id,
                "dataset_source": "vctk",
            })
        except Exception:
            continue
        
        if (i + 1) % 50 == 0:
            gc.collect()
            print(f"  Processed {i+1}/{len(vctk_wavs)}...", end="\r")

else:
    print("No VCTK data available. TTS training will use LJSpeech only.")
    print("(This limits speaker diversity but still works.)")

print(f"\nVCTK TTS records: {len(tts_records)}")
gc.collect()
print(f"Free disk: {disk_free_gb():.1f} GB")

No VCTK data available. TTS training will use LJSpeech only.
(This limits speaker diversity but still works.)

VCTK TTS records: 0
Free disk: 28.2 GB


---
## 2.4 — Dataset 4: LJSpeech (TTS Baseline)

**What it gives us:** Single speaker, 24h clean English, excellent phoneme coverage  
**Processing:** Already at 22050 Hz — just loudness normalize and filter  
**Note:** We check the existing LJSpeech from the previous pipeline first

In [13]:
# 2.4.1 — Find LJSpeech data
LJ_RAW = RAW_DIR / "ljspeech"
LJ_RAW.mkdir(parents=True, exist_ok=True)

lj_wavs = []
lj_metadata = None
lj_source = None

# Check existing LJSpeech locations
for lj_check in [LJ_RAW, BASE.parent / "data" / "raw" / "ljspeech"]:
    if lj_check.exists():
        # Look for wavs directory
        wav_dir = lj_check / "wavs"
        if wav_dir.exists():
            lj_wavs = sorted(wav_dir.glob("*.wav"))
        else:
            lj_wavs = sorted(lj_check.rglob("*.wav"))
        
        # Look for metadata.csv
        for meta_name in ["metadata.csv", "metadata.txt"]:
            meta_path = lj_check / meta_name
            if meta_path.exists():
                lj_metadata = meta_path
        
        if len(lj_wavs) > 0:
            lj_source = lj_check
            break

if len(lj_wavs) == 0:
    print("LJSpeech not found locally. Attempting download...")
    try:
        from datasets import load_dataset
        lj_ds = load_dataset("lj_speech", split="train", trust_remote_code=True)
        print(f"  ✓ LJSpeech loaded: {len(lj_ds)} samples")
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        print("  Manual download: https://keithito.com/LJ-Speech-Dataset/")
        print(f"  Extract to: {LJ_RAW}")
        lj_ds = None
else:
    lj_ds = None
    print(f"✓ LJSpeech: {len(lj_wavs)} files from {lj_source}")
    if lj_metadata:
        print(f"  Metadata: {lj_metadata}")

✓ LJSpeech: 13100 files from /home/nibiru/Documents/sem6project/Speech2/data/raw/ljspeech
  Metadata: /home/nibiru/Documents/sem6project/Speech2/data/raw/ljspeech/metadata.csv


In [14]:
# 2.4.2 — Process LJSpeech → 22050 Hz, loudness normalize

# Load LJSpeech metadata (pipe-delimited: ID|transcript|normalized)
lj_texts = {}
if lj_metadata:
    with open(lj_metadata, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) >= 2:
                lj_texts[parts[0].strip()] = parts[-1].strip()  # use normalized text
    print(f"Loaded {len(lj_texts)} LJSpeech transcripts")

lj_tts_records = []

if lj_ds is not None:
    # ─── From HuggingFace ───
    print(f"Processing {len(lj_ds)} LJSpeech samples...")
    for i, sample in enumerate(lj_ds):
        try:
            audio = sample["audio"]
            wav = np.array(audio["array"], dtype=np.float32)
            sr = audio["sampling_rate"]
            wav = normalize_audio(wav, sr, 22050)
            duration = len(wav) / 22050
            
            if duration < 1.0 or duration > 10.0:
                continue
            
            transcript = sample.get("normalized_text", sample.get("text", ""))
            fname = f"lj_{i:06d}.wav"
            out_path = TTS_OUT / fname
            save_wav(wav, out_path, 22050)
            
            lj_tts_records.append({
                "audio_path": str(out_path.relative_to(BASE)),
                "duration_sec": round(duration, 2),
                "transcript": transcript,
                "speaker_id": "LJ",
                "dataset_source": "ljspeech",
            })
        except Exception:
            continue
        
        if (i + 1) % 50 == 0:
            gc.collect()
            print(f"  Processed {i+1}/{len(lj_ds)}...", end="\r")
    
    del lj_ds
    gc.collect()

elif len(lj_wavs) > 0:
    # ─── From local files ───
    print(f"Processing {len(lj_wavs)} LJSpeech files...")
    for i, wav_path in enumerate(lj_wavs):
        try:
            wav, sr = sf.read(str(wav_path))
            wav = normalize_audio(wav, sr, 22050)
            duration = len(wav) / 22050
            
            if duration < 1.0 or duration > 10.0:
                continue
            
            transcript = lj_texts.get(wav_path.stem, "")
            fname = f"lj_{i:06d}.wav"
            out_path = TTS_OUT / fname
            save_wav(wav, out_path, 22050)
            
            lj_tts_records.append({
                "audio_path": str(out_path.relative_to(BASE)),
                "duration_sec": round(duration, 2),
                "transcript": transcript,
                "speaker_id": "LJ",
                "dataset_source": "ljspeech",
            })
        except Exception:
            continue
        
        if (i + 1) % 50 == 0:
            gc.collect()
            print(f"  Processed {i+1}/{len(lj_wavs)}...", end="\r")

else:
    print("No LJSpeech data available.")

print(f"\nLJSpeech TTS records: {len(lj_tts_records)}")
gc.collect()
print(f"Free disk: {disk_free_gb():.1f} GB")

Loaded 13100 LJSpeech transcripts
Processing 13100 LJSpeech files...
  Processed 13100/13100...
LJSpeech TTS records: 12910
Free disk: 23.6 GB


In [15]:
# 2.4.3 — Build combined TTS manifest (VCTK + LJSpeech)
import pandas as pd

all_tts = tts_records + lj_tts_records

if len(all_tts) > 0:
    tts_df = pd.DataFrame(all_tts)
    
    # Train/val/test split (90/5/5 by percentage)
    np.random.seed(42)
    n = len(tts_df)
    indices = np.random.permutation(n)
    train_end = int(0.9 * n)
    val_end = int(0.95 * n)
    
    splits = np.array(["train"] * n)
    splits[indices[train_end:val_end]] = "val"
    splits[indices[val_end:]] = "test"
    tts_df["split"] = splits
    
    tts_manifest_path = META_DIR / "tts_manifest.csv"
    tts_df.to_csv(tts_manifest_path, index=False)
    
    print(f"TTS manifest saved: {tts_manifest_path}")
    print(f"Total rows: {len(tts_df)}")
    print(f"\nBy dataset:")
    print(tts_df["dataset_source"].value_counts())
    print(f"\nBy split:")
    print(tts_df["split"].value_counts())
    print(f"\nUnique speakers: {tts_df['speaker_id'].nunique()}")
    print(f"Total hours: {tts_df['duration_sec'].sum() / 3600:.1f}h")
else:
    print("⚠ No TTS data available.")
    print("  Download VCTK and/or LJSpeech and re-run this section.")

TTS manifest saved: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/metadata/tts_manifest.csv
Total rows: 12910

By dataset:
dataset_source
ljspeech    12910
Name: count, dtype: int64

By split:
split
train    11619
test       646
val        645
Name: count, dtype: int64

Unique speakers: 1
Total hours: 23.4h


---
## 2.5 — Build Emotion Manifest (from RAVDESS)

We use RAVDESS for emotion VAD labels since it has annotated emotions.  
RAVDESS filename encodes: modality-vocal_channel-emotion-intensity-statement-repetition-actor

In [16]:
# 2.5.1 — Process RAVDESS for emotion detection
EMO_OUT = PROCESSED_DIR / "emotion_train"

# RAVDESS emotion → VAD mapping (approximate)
EMOTION_VAD = {
    1: {"emotion": "neutral",   "valence": 0.0,  "arousal": 0.0,  "dominance": 0.0},
    2: {"emotion": "calm",      "valence": 0.3,  "arousal": -0.5, "dominance": 0.0},
    3: {"emotion": "happy",     "valence": 0.8,  "arousal": 0.6,  "dominance": 0.5},
    4: {"emotion": "sad",       "valence": -0.7, "arousal": -0.3, "dominance": -0.5},
    5: {"emotion": "angry",     "valence": -0.6, "arousal": 0.8,  "dominance": 0.7},
    6: {"emotion": "fearful",   "valence": -0.6, "arousal": 0.7,  "dominance": -0.5},
    7: {"emotion": "disgust",   "valence": -0.7, "arousal": 0.3,  "dominance": 0.3},
    8: {"emotion": "surprised", "valence": 0.2,  "arousal": 0.8,  "dominance": 0.0},
}

# Find RAVDESS
ravdess_dir = None
for check_dir in [RAW_DIR / "ravdess", BASE.parent / "data" / "raw" / "ravdess"]:
    if check_dir.exists():
        ravdess_dir = check_dir
        break

emo_records = []
if ravdess_dir:
    rav_wavs = sorted(ravdess_dir.rglob("*.wav"))
    print(f"Processing {len(rav_wavs)} RAVDESS files for emotion...")
    
    for i, wav_path in enumerate(rav_wavs):
        try:
            # Parse RAVDESS filename: 03-01-05-02-01-01-12.wav
            parts = wav_path.stem.split("-")
            if len(parts) < 7:
                continue
            
            emotion_code = int(parts[2])
            actor_id = int(parts[6])
            
            if emotion_code not in EMOTION_VAD:
                continue
            
            vad = EMOTION_VAD[emotion_code]
            
            wav, sr = sf.read(str(wav_path))
            wav = normalize_audio(wav, sr, 16000)
            duration = len(wav) / 16000
            
            if duration < 1.0 or duration > 10.0:
                continue
            
            fname = f"emo_{i:06d}.wav"
            out_path = EMO_OUT / fname
            save_wav(wav, out_path, 16000)
            
            emo_records.append({
                "audio_path": str(out_path.relative_to(BASE)),
                "duration_sec": round(duration, 2),
                "emotion": vad["emotion"],
                "valence": vad["valence"],
                "arousal": vad["arousal"],
                "dominance": vad["dominance"],
                "speaker_id": f"Actor_{actor_id:02d}",
                "dataset_source": "ravdess",
            })
        except Exception:
            continue
        
        if (i + 1) % 50 == 0:
            gc.collect()
    
    print(f"✓ Emotion records: {len(emo_records)}")
else:
    print("⚠ RAVDESS not found. Emotion detector training will be limited.")

Processing 1440 RAVDESS files for emotion...
✓ Emotion records: 1435


In [17]:
# 2.5.2 — Save emotion manifest
import pandas as pd

if len(emo_records) > 0:
    emo_df = pd.DataFrame(emo_records)
    
    # Split 80/10/10 by speaker
    np.random.seed(42)
    emo_speakers = emo_df["speaker_id"].unique()
    np.random.shuffle(emo_speakers)
    n = len(emo_speakers)
    train_spks = set(emo_speakers[:int(0.8*n)])
    val_spks = set(emo_speakers[int(0.8*n):int(0.9*n)])
    
    emo_df["split"] = emo_df["speaker_id"].apply(
        lambda s: "train" if s in train_spks else ("val" if s in val_spks else "test")
    )
    
    emo_manifest_path = META_DIR / "emotion_manifest.csv"
    emo_df.to_csv(emo_manifest_path, index=False)
    
    print(f"Emotion manifest saved: {emo_manifest_path}")
    print(f"Total rows: {len(emo_df)}")
    print(f"\nEmotion distribution:")
    print(emo_df["emotion"].value_counts())
    print(f"\nSplit distribution:")
    print(emo_df["split"].value_counts())
else:
    print("No emotion records to save.")

Emotion manifest saved: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/metadata/emotion_manifest.csv
Total rows: 1435

Emotion distribution:
emotion
sad          192
angry        192
disgust      192
happy        191
fearful      191
surprised    191
calm         190
neutral       96
Name: count, dtype: int64

Split distribution:
split
train    1137
test      178
val       120
Name: count, dtype: int64


---
## 2.6 — Pre-Extract Speaker Embeddings

Before TTS training, extract 192-dim ECAPA-TDNN embeddings for every TTS clip.  
This avoids computing embeddings on-the-fly during training (saves GPU memory + time).

**Storage:** ~45k clips × 192 floats × 4 bytes ≈ 35 MB. Negligible.

In [18]:
# 2.6.1 — Load pretrained ECAPA-TDNN speaker encoder
from speechbrain.inference.speaker import EncoderClassifier

print("Loading ECAPA-TDNN speaker encoder (CPU)...")
spk_encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=str(BASE / "checkpoints" / "speaker_encoder" / "pretrained"),
    run_opts={"device": "cpu"},
)
print("✓ Speaker encoder loaded on CPU.")

Loading ECAPA-TDNN speaker encoder (CPU)...
✓ Speaker encoder loaded on CPU.


In [19]:
# 2.6.2 — Extract embeddings for all TTS training clips
import pandas as pd

tts_manifest_path = META_DIR / "tts_manifest.csv"
if tts_manifest_path.exists():
    tts_df = pd.read_csv(tts_manifest_path)
    print(f"Extracting embeddings for {len(tts_df)} TTS clips...")
    
    emb_map = {}  # audio_path → embedding_path
    
    for i, row in tts_df.iterrows():
        audio_path = BASE / row["audio_path"]
        if not audio_path.exists():
            continue
        
        try:
            wav, sr = torchaudio.load(str(audio_path))
            # Resample to 16kHz for speaker encoder if needed
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            
            # Take first 5 seconds max
            max_samples = 5 * 16000
            if wav.shape[1] > max_samples:
                wav = wav[:, :max_samples]
            
            # Extract embedding
            with torch.no_grad():
                embedding = spk_encoder.encode_batch(wav)
                embedding = embedding.squeeze().cpu().numpy()
                # L2 normalize
                embedding = embedding / (np.linalg.norm(embedding) + 1e-8)
            
            # Save as .npy
            emb_fname = f"emb_{i:06d}.npy"
            emb_path = EMB_DIR / emb_fname
            np.save(emb_path, embedding)
            
            emb_map[row["audio_path"]] = str(emb_path.relative_to(BASE))
        except Exception:
            continue
        
        if (i + 1) % 100 == 0:
            gc.collect()
            print(f"  {i+1}/{len(tts_df)}...", end="\r")
    
    # Add embedding paths to manifest
    tts_df["embedding_path"] = tts_df["audio_path"].map(emb_map).fillna("")
    tts_df.to_csv(tts_manifest_path, index=False)
    
    n_emb = sum(1 for v in emb_map.values() if v)
    emb_size_mb = sum(os.path.getsize(BASE / p) for p in emb_map.values() if p) / 1e6
    print(f"\n✓ Extracted {n_emb} embeddings ({emb_size_mb:.1f} MB total)")
    print(f"  Updated TTS manifest with embedding_path column.")
else:
    print("No TTS manifest found — skipping embedding extraction.")
    print("Run TTS data processing cells first.")

Extracting embeddings for 12910 TTS clips...
  12900/12910...
✓ Extracted 12910 embeddings (11.6 MB total)
  Updated TTS manifest with embedding_path column.


In [20]:
# Cleanup speaker encoder from memory
del spk_encoder
gc.collect()
torch.cuda.empty_cache()
print("✓ Speaker encoder unloaded.")

✓ Speaker encoder unloaded.


---
## 2.7 — Raw Data Cleanup

Delete raw dataset files to free disk space.  
Only run this AFTER verifying all processed data is correct.

In [21]:
# 2.7.1 — Summary of all manifests before cleanup
import pandas as pd

print("="*60)
print("DATA PIPELINE SUMMARY")
print("="*60)

manifests = {
    "ASR":        META_DIR / "asr_manifest.csv",
    "Speaker":    META_DIR / "speaker_manifest.csv",
    "Translation":META_DIR / "translation_pairs.csv",
    "TTS":        META_DIR / "tts_manifest.csv",
    "Emotion":    META_DIR / "emotion_manifest.csv",
}

for name, path in manifests.items():
    if path.exists():
        df = pd.read_csv(path)
        hours = df["duration_sec"].sum() / 3600 if "duration_sec" in df.columns else 0
        print(f"  {name:<12s}: {len(df):>6d} rows | {hours:>5.1f} hours | {path.name}")
    else:
        print(f"  {name:<12s}: NOT FOUND")

print(f"\nFree disk: {disk_free_gb():.1f} GB")
print("="*60)

DATA PIPELINE SUMMARY
  ASR         :   2949 rows |   9.4 hours | asr_manifest.csv
  Speaker     :   1435 rows |   1.5 hours | speaker_manifest.csv
  Translation :   1719 rows |   0.0 hours | translation_pairs.csv
  TTS         :  12910 rows |  23.4 hours | tts_manifest.csv
  Emotion     :   1435 rows |   1.5 hours | emotion_manifest.csv

Free disk: 28.6 GB


In [22]:
# 2.7.2 — Delete raw data (OPTIONAL — only run when you're confident)
# Uncomment the lines below to free disk space

# import shutil
# raw_dirs = list(RAW_DIR.iterdir())
# for d in raw_dirs:
#     if d.is_dir():
#         size_mb = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e6
#         print(f"  Deleting {d.name}/ ({size_mb:.0f} MB)...")
#         shutil.rmtree(d)
# print(f"\n✓ Raw data deleted. Free disk: {disk_free_gb():.1f} GB")

print("Raw data cleanup is COMMENTED OUT for safety.")
print("Uncomment the lines above after verifying all processed data is correct.")

Raw data cleanup is COMMENTED OUT for safety.
Uncomment the lines above after verifying all processed data is correct.


---
## ✓ Notebook 02 Complete

**What we accomplished:**
- Downloaded/found CoVoST-2 (or FLEURS fallback) → ASR manifest
- Processed Rasa/RAVDESS → Speaker encoder manifest
- Processed VCTK → TTS manifest (multi-speaker)
- Processed LJSpeech → TTS manifest (single speaker, appended)
- Built emotion manifest from RAVDESS with VAD labels
- Pre-extracted speaker embeddings for all TTS clips
- Built translation pairs manifest

**Next:** Open `03_whisper_asr_finetuning.ipynb` to fine-tune Whisper on Telugu.